In [1]:
# Cell 1: Import libraries
import requests
from bs4 import BeautifulSoup

In [2]:

# Cell 2: Get HTML from a URL with headers
url = "https://www.espn.com/college-football/scoreboard/_/week/14/year/2025/seasontype/2/group/80"

# Headers to make the request look like a real browser
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

response = requests.get(url, headers=headers)
html = response.text

print(f"Status code: {response.status_code}")
print(f"HTML length: {len(html)} characters")

Status code: 200
HTML length: 1782025 characters


In [3]:
# Cell 3: Parse HTML with BeautifulSoup
soup = BeautifulSoup(html, 'html.parser')

print(f"Page title: {soup.title.string}")
print("HTML parsed successfully!")

Page title: College Football Scores Week 14, FBS 2025 Regular Season - ESPN
HTML parsed successfully!


In [4]:
fbs_schools = ['Army', 'Tulane', 'Navy', 'Memphis', 'East Carolina',
               'South Florida','UTSA', 'Charlotte', 'North Texas', 'Rice', 'Temple', 'UAB',
               'Florida Atlantic', 'San José State', 'Tulsa',
               'SMU', 'Clemson', 'Miami', 'Syracuse', 'Louisville',
               'Georgia Tech', 'Duke', 'Virginia Tech', 'Boston College',
               'Pittsburgh', 'NC State', 'North Carolina', 'Virginia',
               'California', 'Wake Forest', 'Stanford', 'Florida State',
               'Notre Dame', 'Arizona State', 'BYU', 'Iowa State',
               'Colorado', 'Baylor', 'TCU', 'Texas Tech', 'Kansas State',
               'West Virginia', 'Kansas', 'Cincinnati', 'Houston', 'Utah',
               'UCF', 'Arizona', 'Oklahoma State', 'Oregon', 'Indiana',
               'Penn State', 'Ohio State', 'Illinois', 'Iowa', 'Michigan',
               'Minnesota', 'Rutgers', 'Washington', 'USC', 'UCLA', 'Nebraska',
               'Michigan State', 'Wisconsin', 'Northwestern', 'Maryland', 'Purdue',
               'Jacksonville State', 'Western Kentucky', 'Sam Houston', 'Liberty', 'Louisiana Tech',
               'UTEP', 'Florida International', 'New Mexico State', 'Middle Tennessee',
               'Kennesaw State', 'Delaware', 'Missouri State', 'UConn',
               'Massachusetts', 'Miami (OH)', 'Ohio', 'Buffalo', 'Bowling Green',
               'Western Michigan', 'Toledo', 'Northern Illinois', 'Akron',
               'Eastern Michigan', 'Central Michigan', 'Ball State', 'Kent State',
               'Boise State', 'UNLV', 'Colorado State', 'Fresno State',
               'New Mexico', 'Utah State', "Hawai'i", 'Air Force', 'San Diego State',
               'Wyoming', 'Nevada', 'Oregon State', 'Washington State',
               'Texas', 'Georgia', 'Tennessee', 'Alabama', 'South Carolina',
               'Texas A&M', 'LSU', 'Ole Miss', 'Missouri', 'Florida',
               'Arkansas', 'Vanderbilt', 'Oklahoma', 'Auburn', 'Kentucky',
               'Mississippi State', 'Marshall', 'Georgia Southern', 'James Madison',
               'Old Dominion', 'Coastal Carolina', 'App State', 'Georgia State',
               'Louisiana', 'Texas State', 'Arkansas State', 'South Alabama',
               'UL Monroe', 'Troy', 'Southern Miss']


In [5]:
import pandas as pd

mydf = pd.DataFrame(columns=['TeamName','Score','Home-Away'])

In [6]:
# Cell 4: Basic exploration
# Find all paragraphs
all_names = soup.find_all('div', class_='ScoreCell__TeamName')
print(f"Found {len(all_names)} divs with that class")
all_scores = soup.find_all('div', class_='ScoreCell__Score')
print(f"Found {len(all_scores)} divs with that class")


Found 134 divs with that class
Found 134 divs with that class


In [7]:
mydf['TeamName'] = [name.text for name in all_names]
mydf['Score'] = [score.text for score in all_scores]
mydf['Home-Away'] = ['Away', 'Home'] * int(len(all_names)/2)

In [8]:
formatted_df = pd.DataFrame(columns = ['Team1', 'Team2', 'Result', 'Margin', 'Road'])

In [9]:
for i in range(len(mydf)):
    if mydf.iloc[i].TeamName not in fbs_schools:
        name = 'FCS'
    else:
        name = mydf.iloc[i].TeamName
    if i%2 == 0:
        formatted_df.loc[i, 'Team1'] = name
        if mydf.iloc[i+1].TeamName not in fbs_schools:
            formatted_df.loc[i, 'Team2'] = 'FCS'
        else:
            formatted_df.loc[i, 'Team2'] = mydf.iloc[i+1].TeamName
        if int(mydf.iloc[i].Score) > int(mydf.iloc[i+1].Score):
            formatted_df.loc[i, 'Result'] = 'WIN'
        else:
            formatted_df.loc[i, 'Result'] = 'LOSS'
        formatted_df.loc[i, 'Margin'] = abs(int(mydf.iloc[i].Score) - int(mydf.iloc[i+1].Score))
        formatted_df.loc[i, 'Road'] = 'Road'
    else:
        formatted_df.loc[i, 'Team1'] = name
        if mydf.iloc[i-1].TeamName not in fbs_schools:
            formatted_df.loc[i, 'Team2'] = 'FCS'
        else:
            formatted_df.loc[i, 'Team2'] = mydf.iloc[i-1].TeamName
        if int(mydf.iloc[i].Score) > int(mydf.iloc[i-1].Score):
            formatted_df.loc[i, 'Result'] = 'WIN'
        else:
            formatted_df.loc[i, 'Result'] = 'LOSS'
        formatted_df.loc[i, 'Margin'] = abs(int(mydf.iloc[i].Score) - int(mydf.iloc[i-1].Score))
        formatted_df.loc[i, 'Road'] = 'Home'

final_df = formatted_df[formatted_df.Team1 != 'FCS'].reset_index(drop=True)

In [10]:
import xlwings as xw


# Connect to the active workbook that's already open
wb = xw.books.active  # Connect to specific open workbook
print(wb)

# Get a worksheet
ws = wb.sheets['Week 15']  # or wb.sheets[0] for first sheet
(print(ws))


ws.range('A2').value = final_df.values  # Put DataFrame starting at A2

print("Excel updated successfully!")


<Book [Simplified CFB Rankings - 2025 - Team Points.xlsx]>
<Sheet [Simplified CFB Rankings - 2025 - Team Points.xlsx]Week 15>
Excel updated successfully!
